# Blinkit Sales Analytics



## 0. Imports and paths

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

folder_path = 'C:/Users/HP/Downloads/blinkit_sales_data'
export_path = os.path.join(folder_path, 'powerbi')
os.makedirs(export_path, exist_ok=True)

In [2]:
# Convert the two Excel icon files to CSV (skipped if a file is missing)
for f in ['Category_Icons.xlsx', 'Rating_Icon.xlsx']:
    file_path = os.path.join(folder_path, f)
    if os.path.exists(file_path):
        pd.read_excel(file_path).to_csv(os.path.join(folder_path, f.replace('.xlsx', '.csv')), index=False)
        print('Converted:', f)

Converted: Category_Icons.xlsx
Converted: Rating_Icon.xlsx


## 1. Load all tables

In [3]:
blinkit_customers = pd.read_csv(os.path.join(folder_path, 'blinkit_customers.csv'))
blinkit_cust_feedback = pd.read_csv(os.path.join(folder_path, 'blinkit_customer_feedback.csv'))
blinkit_delivery_perfo = pd.read_csv(os.path.join(folder_path, 'blinkit_delivery_performance.csv'))
blinkit_inventory = pd.read_csv(os.path.join(folder_path, 'blinkit_inventory.csv'))
blinkit_inventory_new = pd.read_csv(os.path.join(folder_path, 'blinkit_inventoryNew.csv'))
blinkit_marketing_perfo = pd.read_csv(os.path.join(folder_path, 'blinkit_marketing_performance.csv'))
blinkit_orders = pd.read_csv(os.path.join(folder_path, 'blinkit_orders.csv'))
blinkit_order_items = pd.read_csv(os.path.join(folder_path, 'blinkit_order_items.csv'))
blinkit_products = pd.read_csv(os.path.join(folder_path, 'blinkit_products.csv'))

for name, t in [('customers', blinkit_customers), ('feedback', blinkit_cust_feedback),
                ('delivery', blinkit_delivery_perfo), ('inventory (old)', blinkit_inventory),
                ('inventory (new)', blinkit_inventory_new), ('marketing', blinkit_marketing_perfo),
                ('orders', blinkit_orders), ('order items', blinkit_order_items), ('products', blinkit_products)]:
    print(f'{name:16}', t.shape)

customers        (2500, 11)
feedback         (5000, 8)
delivery         (5000, 8)
inventory (old)  (75172, 4)
inventory (new)  (18105, 4)
marketing        (5400, 11)
orders           (5000, 10)
order items      (5000, 4)
products         (268, 10)


## 2. Clean customers

In [4]:
# data types
blinkit_customers['registration_date'] = pd.to_datetime(blinkit_customers['registration_date'])
blinkit_customers['phone'] = blinkit_customers['phone'].astype(str)
blinkit_customers['customer_name'] = blinkit_customers['customer_name'].astype(str)
blinkit_customers['customer_segment'] = blinkit_customers['customer_segment'].astype('category')

# customer_id: pad with zeros to 8 digits
blinkit_customers['customer_id'] = blinkit_customers['customer_id'].astype(str).str.zfill(8)

# phone flag (valid phone has 10 digits)
blinkit_customers['phone_clean'] = blinkit_customers['phone'].str.replace(r'\D', '', regex=True)
blinkit_customers['phone_flag'] = blinkit_customers['phone_clean'].str.len() != 10

# average order value: float to int
blinkit_customers['avg_order_value'] = blinkit_customers['avg_order_value'].round().astype(int)

# email flag (duplicate emails)
blinkit_customers['email_flag'] = blinkit_customers['email'].duplicated(keep=False)

In [5]:
# pincode: remove '.0', pad digits to 6, flag and blank anything still not 6 characters
blinkit_customers['pincode'] = (blinkit_customers['pincode'].astype(str)
                                .str.replace(r'\.0$', '', regex=True).str.strip())
digits = blinkit_customers['pincode'].str.fullmatch(r'\d+', na=False)
blinkit_customers.loc[digits, 'pincode'] = blinkit_customers.loc[digits, 'pincode'].str.zfill(6)

blinkit_customers['pincode_flag'] = blinkit_customers['pincode'].fillna('').str.len() != 6
blinkit_customers.loc[blinkit_customers['pincode_flag'], 'pincode'] = None

print(blinkit_customers['pincode_flag'].sum(), 'pincodes flagged')
print(blinkit_customers['registration_date'].dt.year.value_counts())

0 pincodes flagged
registration_date
2024    1337
2023    1163
Name: count, dtype: int64


In [6]:
blinkit_customers.head()

,customer_id,customer_name,email,phone,address,area,pincode,registration_date,customer_segment,total_orders,avg_order_value,phone_clean,phone_flag,email_flag,pincode_flag
0,97475543,Niharika Nagi,ektataneja@example.org,912987579691,"23, Nayar Path, Bihar Sharif-154625",Udupi,321865,2023-05-13,Premium,13,452,912987579691,True,False,False
1,22077605,Megha Sachar,vedant45@example.com,915123179717,"51/302, Buch Chowk\nSrinagar-570271",Aligarh,149394,2024-06-18,Inactive,4,825,915123179717,True,False,False
2,47822591,Hema Bahri,samiazaan@example.com,910034076149,"941\nAnne Street, Darbhanga 186125",Begusarai,621411,2024-09-25,Regular,17,1970,910034076149,True,False,False
3,79726146,Zaitra Vig,ishanvi87@example.org,916264232390,"43/94, Ghosh, Alappuzha 635655",Kozhikode,826054,2023-10-04,New,4,220,916264232390,True,False,False
4,57102800,Januja Verma,atideshpande@example.org,917293526596,"06\nOm, Ambarnath 477463",Ichalkaranji,730539,2024-03-22,Inactive,14,578,917293526596,True,False,False


## 3. Clean products

In [7]:
blinkit_products['product_id'] = blinkit_products['product_id'].astype(str)
blinkit_products[['price', 'mrp']] = blinkit_products[['price', 'mrp']].round(0).astype(int)

In [8]:
blinkit_products.head()

,product_id,product_name,category,brand,price,mrp,margin_percentage,shelf_life_days,min_stock_level,max_stock_level
0,153019,Onions,Fruits & Vegetables,Aurora LLC,948,1264,25.0,3,13,88
1,11422,Potatoes,Fruits & Vegetables,Ramaswamy-Tata,127,170,25.0,3,20,65
2,669378,Potatoes,Fruits & Vegetables,Chadha and Sons,212,283,25.0,3,23,70
3,848226,Tomatoes,Fruits & Vegetables,Barad and Sons,210,279,25.0,3,10,51
4,890623,Onions,Fruits & Vegetables,"Sangha, Nagar and Varty",355,473,25.0,3,27,55


## 4. Clean orders

In [9]:
blinkit_orders['order_id'] = blinkit_orders['order_id'].astype(str)
blinkit_orders['customer_id'] = blinkit_orders['customer_id'].astype(str).str.zfill(8)
blinkit_orders['order_total'] = blinkit_orders['order_total'].round(0).astype(int)

for col in ['order_date', 'promised_delivery_time', 'actual_delivery_time']:
    blinkit_orders[col] = pd.to_datetime(blinkit_orders[col], errors='coerce')
print(blinkit_orders[['order_date', 'promised_delivery_time', 'actual_delivery_time']].isna().sum())

order_date                0
promised_delivery_time    0
actual_delivery_time      0
dtype: int64


In [10]:
# delay in minutes and delivery status (our own definition, replaces the raw status column)
blinkit_orders['delay_minutes'] = (
    blinkit_orders['actual_delivery_time'] - blinkit_orders['promised_delivery_time']
).dt.total_seconds() / 60

def check_status(x):
    if pd.isna(x):
        return None
    if x <= 5:
        return 'On Time'
    elif x <= 15:
        return 'Slightly Delayed'
    return 'Significantly Delayed'

blinkit_orders['delivery_status'] = blinkit_orders['delay_minutes'].apply(check_status)
blinkit_orders['delivery_status'].value_counts(normalize=True) * 100

delivery_status
On Time                  69.40
Slightly Delayed         20.74
Significantly Delayed     9.86
Name: proportion, dtype: float64

In [11]:
blinkit_orders.head()

,order_id,customer_id,order_date,promised_delivery_time,actual_delivery_time,delivery_status,order_total,payment_method,delivery_partner_id,store_id,delay_minutes
0,1961864118,30065862,2024-07-17 08:34:01,2024-07-17 08:52:01,2024-07-17 08:47:01,On Time,3197,Cash,63230,4771,-5.0
1,1549769649,09573071,2024-05-28 13:14:29,2024-05-28 13:25:29,2024-05-28 13:27:29,On Time,977,Cash,14983,7534,2.0
2,9185164487,45477575,2024-09-23 13:07:12,2024-09-23 13:25:12,2024-09-23 13:29:12,On Time,839,UPI,39859,9886,4.0
3,9644738826,88067569,2023-11-24 16:16:56,2023-11-24 16:34:56,2023-11-24 16:33:56,On Time,440,Card,61497,7917,-1.0
4,5427684290,83298567,2023-11-20 05:00:39,2023-11-20 05:17:39,2023-11-20 05:18:39,On Time,2527,Cash,84315,2741,1.0


## 5. Clean order items

In [12]:
blinkit_order_items['order_id'] = blinkit_order_items['order_id'].astype(str)
blinkit_order_items['product_id'] = blinkit_order_items['product_id'].astype(str)
blinkit_order_items['unit_price'] = blinkit_order_items['unit_price'].round(0).astype(int)
blinkit_order_items['item_total'] = blinkit_order_items['quantity'] * blinkit_order_items['unit_price']

# one product per order? (empty result = yes, so no basket analysis possible)
basket = blinkit_order_items.groupby('order_id').agg(
    different_products=('product_id', 'nunique'), total_units=('quantity', 'sum')).reset_index()
print(basket[basket['different_products'] > 1])

Empty DataFrame
Columns: [order_id, different_products, total_units]
Index: []


In [13]:
blinkit_order_items.head()

,order_id,product_id,quantity,unit_price,item_total
0,1961864118,642612,3,517,1551
1,1549769649,378676,1,881,881
2,9185164487,741341,2,924,1848
3,9644738826,561860,1,875,875
4,5427684290,602241,2,977,1954


## 6. Clean delivery performance

In [14]:
blinkit_delivery_perfo['order_id'] = blinkit_delivery_perfo['order_id'].astype(str)
blinkit_delivery_perfo['delivery_partner_id'] = blinkit_delivery_perfo['delivery_partner_id'].astype(str)
blinkit_delivery_perfo['promised_time'] = pd.to_datetime(blinkit_delivery_perfo['promised_time'])
blinkit_delivery_perfo['actual_time'] = pd.to_datetime(blinkit_delivery_perfo['actual_time'])
blinkit_delivery_perfo['delivery_time_minutes'] = blinkit_delivery_perfo['delivery_time_minutes'].astype(int)
blinkit_delivery_perfo['delivery_status'] = blinkit_delivery_perfo['delivery_status'].astype('category')

In [15]:
blinkit_delivery_perfo.head()

,order_id,delivery_partner_id,promised_time,actual_time,delivery_time_minutes,distance_km,delivery_status,reasons_if_delayed
0,1961864118,63230,2024-07-17 08:52:01,2024-07-17 08:47:01,-5,0.96,On Time,NaN
1,1549769649,14983,2024-05-28 13:25:29,2024-05-28 13:27:29,2,0.98,On Time,Traffic
2,9185164487,39859,2024-09-23 13:25:12,2024-09-23 13:29:12,4,3.83,On Time,Traffic
3,9644738826,61497,2023-11-24 16:34:56,2023-11-24 16:33:56,-1,2.76,On Time,NaN
4,5427684290,84315,2023-11-20 05:17:39,2023-11-20 05:18:39,1,2.63,On Time,Traffic


## 7. Build `orders_final` (main table)

In [16]:
orders_final = (blinkit_orders
    .merge(blinkit_order_items[['order_id', 'product_id', 'quantity', 'unit_price', 'item_total']],
           on='order_id', how='left')
    .merge(blinkit_products[['product_id', 'product_name', 'category', 'brand', 'margin_percentage']],
           on='product_id', how='left')
    .merge(blinkit_delivery_perfo[['order_id', 'distance_km', 'reasons_if_delayed']],
           on='order_id', how='left')
    .merge(blinkit_customers[['customer_id', 'area', 'customer_segment']],
           on='customer_id', how='left'))

orders_final['revenue'] = orders_final['item_total']
orders_final = orders_final.drop(columns=['order_total', 'item_total'])



In [17]:
# delivery metrics
orders_final['delivery_time_min'] = (
    orders_final['actual_delivery_time'] - orders_final['order_date']).dt.total_seconds() / 60
orders_final['is_late'] = orders_final['delay_minutes'] > 5
orders_final['order_hour'] = orders_final['order_date'].dt.hour
orders_final['distance_band'] = pd.cut(orders_final['distance_km'], [0, 2, 4, 5.1],
                                       right=False, labels=['<2 km', '2-4 km', '4-5 km'])
orders_final.head()

,order_id,customer_id,order_date,promised_delivery_time,actual_delivery_time,delivery_status,payment_method,delivery_partner_id,store_id,delay_minutes,...,margin_percentage,distance_km,reasons_if_delayed,area,customer_segment,revenue,delivery_time_min,is_late,order_hour,distance_band
0,1961864118,30065862,2024-07-17 08:34:01,2024-07-17 08:52:01,2024-07-17 08:47:01,On Time,Cash,63230,4771,-5.0,...,35.0,0.96,NaN,Allahabad,Regular,1551,13.0,False,8,<2 km
1,1549769649,09573071,2024-05-28 13:14:29,2024-05-28 13:25:29,2024-05-28 13:27:29,On Time,Cash,14983,7534,2.0,...,30.0,0.98,Traffic,Thrissur,New,881,13.0,False,13,<2 km
2,9185164487,45477575,2024-09-23 13:07:12,2024-09-23 13:25:12,2024-09-23 13:29:12,On Time,UPI,39859,9886,4.0,...,20.0,3.83,Traffic,Vellore,Inactive,1848,22.0,False,13,2-4 km
3,9644738826,88067569,2023-11-24 16:16:56,2023-11-24 16:34:56,2023-11-24 16:33:56,On Time,Card,61497,7917,-1.0,...,30.0,2.76,NaN,Gaya,Premium,875,17.0,False,16,2-4 km
4,5427684290,83298567,2023-11-20 05:00:39,2023-11-20 05:17:39,2023-11-20 05:18:39,On Time,Cash,84315,2741,1.0,...,35.0,2.63,Traffic,Asansol,Premium,1954,18.0,False,5,2-4 km


## 8. Clean marketing

In [18]:
campaign_audience = {
    'New User Discount': 'New Users',
    'Membership Drive': 'Premium',
    'Email Campaign': 'Inactive',
    'Weekend Special': 'All',
    'Festival Offer': 'All',
    'Flash Sale': 'All',
    'Category Promotion': 'All',
    'App Push Notification': 'All',
    'Referral Program': 'All',
}
blinkit_marketing_perfo['target_audience'] = blinkit_marketing_perfo['campaign_name'].map(campaign_audience)

blinkit_marketing_perfo['campaign_id'] = blinkit_marketing_perfo['campaign_id'].astype(str)
blinkit_marketing_perfo['date'] = pd.to_datetime(blinkit_marketing_perfo['date'])
blinkit_marketing_perfo['spend'] = blinkit_marketing_perfo['spend'].round(0).astype(int)
blinkit_marketing_perfo['revenue_generated'] = blinkit_marketing_perfo['revenue_generated'].round(0).astype(int)

# return on ad spend and click-through rate
blinkit_marketing_perfo['roas'] = (blinkit_marketing_perfo['revenue_generated'] / blinkit_marketing_perfo['spend']).round(2)
blinkit_marketing_perfo['ctr_pct'] = (blinkit_marketing_perfo['clicks'] / blinkit_marketing_perfo['impressions'] * 100).round(2)

In [19]:
blinkit_marketing_perfo.head()

,campaign_id,campaign_name,date,target_audience,channel,impressions,clicks,conversions,spend,revenue_generated,roas,ctr_pct
0,548299,New User Discount,2024-11-05,New Users,App,3130,163,78,1432,4778,3.34,5.21
1,390914,Weekend Special,2024-11-05,All,App,3925,494,45,4506,6238,1.38,12.59
2,834385,Festival Offer,2024-11-05,All,Email,7012,370,78,4524,2621,0.58,5.28
3,241523,Flash Sale,2024-11-05,All,SMS,1115,579,86,3623,2955,0.82,51.93
4,595111,Membership Drive,2024-11-05,Premium,Email,7172,795,54,2889,8952,3.10,11.08


## 9. Inventory
The **new** (monthly) table is the final one: about 0.24% bad rows, against about 38.9% in the old daily table.
The old table is kept only for validation.

In [20]:
# new table = final
blinkit_inventory_new['product_id'] = blinkit_inventory_new['product_id'].astype(str)

bad_rows = blinkit_inventory_new[blinkit_inventory_new['damaged_stock'] > blinkit_inventory_new['stock_received']]
print(len(bad_rows), 'bad rows of', len(blinkit_inventory_new),
      '=', round(len(bad_rows) / len(blinkit_inventory_new) * 100, 2), '%')

inventory_final = blinkit_inventory_new[
    blinkit_inventory_new['damaged_stock'] <= blinkit_inventory_new['stock_received']].copy()
inventory_final['month'] = pd.to_datetime(inventory_final['date'], format='%b-%y')
inventory_final['damage_rate'] = (
    inventory_final['damaged_stock'] / inventory_final['stock_received'].replace(0, np.nan) * 100).round(2)

print(inventory_final.shape)

44 bad rows of 18105 = 0.24 %
(18061, 6)


In [21]:
# monthly damage trend (sorted by real date)
trend = inventory_final.groupby('month')[['stock_received', 'damaged_stock']].sum()
trend['damage_rate_%'] = (trend['damaged_stock'] / trend['stock_received'] * 100).round(2)
trend

,stock_received,damaged_stock,damage_rate_%
month,,,
2023-03-01,1292,17,1.32
2023-04-01,1413,0,0.00
2023-05-01,1428,31,2.17
2023-06-01,1151,191,16.59
2023-07-01,1076,25,2.32
2023-08-01,995,173,17.39
2023-09-01,1551,0,0.00
2023-10-01,1511,277,18.33
2023-11-01,1837,34,1.85


In [22]:
blinkit_inventory_new.head()

,product_id,date,stock_received,damaged_stock
0,153019,Mar-23,4,1
1,848226,Mar-23,4,1
2,965755,Mar-23,1,0
3,39154,Mar-23,4,0
4,34186,Mar-23,3,1


## 10. Feedback

In [23]:
blinkit_cust_feedback['order_id'] = blinkit_cust_feedback['order_id'].astype(str)
blinkit_cust_feedback['customer_id'] = blinkit_cust_feedback['customer_id'].astype(str).str.zfill(8)
blinkit_cust_feedback['feedback_date'] = pd.to_datetime(blinkit_cust_feedback['feedback_date'])
blinkit_cust_feedback['sentiment'].value_counts(normalize=True) * 100

sentiment
Neutral     34.76
Negative    32.84
Positive    32.40
Name: proportion, dtype: float64

In [24]:
blinkit_cust_feedback.head()

,feedback_id,order_id,customer_id,rating,feedback_text,feedback_category,sentiment,feedback_date
0,2234710,1961864118,30065862,4,"It was okay, nothing special.",Delivery,Neutral,2024-07-17
1,5450964,1549769649,09573071,3,The order was incorrect.,App Experience,Negative,2024-05-28
2,482108,9185164487,45477575,3,"It was okay, nothing special.",App Experience,Neutral,2024-09-23
3,4823104,9644738826,88067569,4,The product met my expectations.,App Experience,Neutral,2023-11-24
4,3537464,5427684290,83298567,3,Product was damaged during delivery.,Delivery,Negative,2023-11-20


## 11. Checks

In [25]:
print('orders_final rows        :', len(orders_final))
print('duplicate order_id        :', orders_final['order_id'].duplicated().sum())
print('feedback matched to orders:', blinkit_cust_feedback['order_id'].isin(orders_final['order_id']).mean().round(3))
print('inventory not in products :', (~inventory_final['product_id'].isin(blinkit_products['product_id'])).sum())
print()
print(orders_final.isna().sum()[lambda s: s > 0])

orders_final rows        : 5000
duplicate order_id        : 0
feedback matched to orders: 1.0
inventory not in products : 0

reasons_if_delayed    1902
dtype: int64


## 12. Quick analysis
Late % is about the same in every distance band, and the correlation between distance and delay is about 0.
So distance is **not** a driver of late deliveries in this data.

In [26]:
print(orders_final['delivery_status'].value_counts(normalize=True) * 100)
print('avg delivery time (min):', orders_final['delivery_time_min'].mean().round(1))
print()
print(orders_final.groupby('distance_band', observed=True)['is_late'].mean() * 100)
print()
print(orders_final[['distance_km', 'delay_minutes']].corr())

delivery_status
On Time                  69.40
Slightly Delayed         20.74
Significantly Delayed     9.86
Name: proportion, dtype: float64
avg delivery time (min): 19.4

distance_band
<2 km     31.138070
2-4 km    29.876656
4-5 km    31.203704
Name: is_late, dtype: float64

               distance_km  delay_minutes
distance_km       1.000000       0.004112
delay_minutes     0.004112       1.000000


In [27]:
print(orders_final['reasons_if_delayed'].value_counts())
print()
print((orders_final.groupby('area')['is_late'].mean() * 100).sort_values(ascending=False).head(10))

reasons_if_delayed
Traffic    3098
Name: count, dtype: int64

area
Rourkela       100.000000
Raipur          80.000000
Phusro          75.000000
Jalandhar       75.000000
Kolhapur        72.727273
Raichur         66.666667
Siwan           62.500000
Hazaribagh      61.538462
Rajahmundry     61.111111
Ujjain          60.000000
Name: is_late, dtype: float64


## 13. Export for Power BI
The tables above are unchanged. Extra columns and the removal of personal data apply only to the export copies.

In [28]:
orders_export = orders_final.copy()
orders_export['order_day'] = orders_export['order_date'].dt.normalize()
orders_export['is_late_flag'] = orders_export['is_late'].astype(int)

customers_export = blinkit_customers.drop(columns=['phone', 'phone_clean', 'email'], errors='ignore')

tables = {
    'orders_final': orders_export,
    'customers': customers_export,
    'products': blinkit_products,
    'feedback': blinkit_cust_feedback,
    'marketing': blinkit_marketing_perfo,
    'inventory': inventory_final,
    'delivery': blinkit_delivery_perfo,
}
for name, t in tables.items():
    t.to_csv(os.path.join(export_path, name + '.csv'), index=False)
    print(name, t.shape)

orders_final (5000, 28)
customers (2500, 12)
products (268, 10)
feedback (5000, 8)
marketing (5400, 12)
inventory (18061, 6)
delivery (5000, 8)
